In [1]:
%pip install -qU langchain langchain-ollama langgraph pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
from langchain_core.tools import tool

caminho_prompt = "../prompts/system_prompt.md"

try:
    with open(caminho_prompt, "r", encoding="utf-8") as arquivo:
        SYSTEM_PROMPT = arquivo.read()
    print("System Prompt carregado")
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado. Verifique o caminho '{caminho_prompt}'")

# Simulando APIs reais
@tool
def buscar_dados_wearable(patient_id: str) -> str:
    """Busca os últimos dados vitais sincronizados do smartwatch (Apple Health/Google Fit) do paciente. Exemplo de patient_id: 9988"""
    print(f"\n[TOOL EXECUTADA: Buscando dados do wearable para o paciente ID {patient_id}...]")
    return f"Dados do Wearable para {patient_id}: Frequência Cardíaca em repouso 110 bpm. SpO2: 98%."

@tool
def buscar_historico_paciente(patient_id: str) -> str:
    """Busca o histórico médico e receitas de uso contínuo do paciente."""
    print(f"\n[TOOL EXECUTADA: Acessando prontuário do paciente ID {patient_id}...]")
    return f"Prontuário de {patient_id}. Medicações contínuas: Losartana 50mg. Alergias: Nenhuma."

tools = [buscar_dados_wearable, buscar_historico_paciente]

System Prompt carregado


In [6]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage

# 1. garante memoria da conversa
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. iniciar llama (verifique se llama esta rodando localmente)
llm = ChatOllama(model="llama3.1", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# 3. chatbot
def chatbot(state: State):
    mensagens_com_sys_prompt = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resposta = llm_with_tools.invoke(mensagens_com_sys_prompt)
    return {"messages": [resposta]}

# 4. orquestração LangGraph
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# rotas
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition) # tools_condition avalia se o LLM pediu para usar uma tool. 
graph_builder.add_edge("tools", "chatbot") # se sim, vai para o nó "tools". se não, encerra a execução e devolve para o usuário.

# 5. grafo com MemorySaver
memory = MemorySaver()
app = graph_builder.compile(checkpointer=memory)

print("Orquestração LangGraph com memória compilada")

Orquestração LangGraph com memória compilada


In [7]:
def conversar(mensagem, thread_id="sessao_01"):
    print(f"Paciente: {mensagem}")
    
    # thread_id é a chave que diz ao LangGraph para buscar a memória desta conversa específica
    config = {"configurable": {"thread_id": thread_id}}
    
    for evento in app.stream({"messages": [("user", mensagem)]}, config, stream_mode="values"):
        ultima_mensagem = evento["messages"][-1]
        
    print(f"BluaDiagnostics: {ultima_mensagem.content}\n")
    print("-" * 70 + "\n")

print("=== INICIANDO DEMONSTRAÇÃO DO BLUADIAGNOSTICS ===\n")

# 1: forçar o LLM a chamar a tool buscar_dados_wearable
conversar("Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.")

#  2: testando a memória e continuidade clínica (LLM deve lembrar dos 110 bpm)
conversar("Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.")

# 3: testando as regras de limite de escopo do System Prompt (Out of Scope)
conversar("Entendi. Mudando de assunto, por que o valor do meu plano de saúde aumentou 15% neste mês? Quero contestar a fatura.")

=== INICIANDO DEMONSTRAÇÃO DO BLUADIAGNOSTICS ===

Paciente: Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.

[TOOL EXECUTADA: Buscando dados do wearable para o paciente ID 9988...]
BluaDiagnostics: * Você mencionou que está se sentindo normal apesar da frequência cardíaca elevada.
* A sua frequência cardíaca em repouso é de 110 bpm, o que pode ser considerado alto para alguém que não esteja realizando atividade física intensa.
* O seu SpO2 está dentro do intervalo normal.

Além da dor de cabeça, você notou febre ou náusea?

----------------------------------------------------------------------

Paciente: Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.

[TOOL EXECUTADA: Buscando dados do wearable para o paciente ID 9988...]
BluaDiagnostics: * Você mencionou que não tomou café hoje e não passou por estresse, o que pode ajudar a explicar a frequência cardía